# 🧠 Model Pipeline – Encoding Chaining Dataset (Cross-Subject)
**File:** `encoding_chaining.csv`  
**Task:** 9-class EEG Scenario Classification  
**Protocol:** Cross-subject evaluation using **GroupKFold** by `subject_id`  
**Output folder:** `encoding_model_outputs/`

> The Encoding Chaining dataset contains the `chain_ratio` feature per  
> `(subject, scenario, task, channel, subband, feature)`. We **pivot** to wide format:  
> one row per `(subject_id, scenario)` with all `task_channel_subband_feature` combinations  
> as columns. This gives a feature-rich representation for classification.


## 0. Setup

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.decomposition import PCA

OUTPUT_DIR = "encoding_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
SCENARIO_LABELS = {
    0:"Lift L.Hand", 1:"Lift R.Hand", 2:"Lift L.Leg",  3:"Lift R.Leg",
    4:"Open Mouth",  5:"Nod Head",    6:"Shake Head",   7:"Want Water", 8:"Use Bathroom"
}
LABELS_S = [f"S{i+1}" for i in range(9)]
CMAPS    = ['Blues','Greens','Oranges','Reds','Purples','YlOrBr','GnBu','RdPu','BuPu']
print("Setup complete.")


## 1. Load & Explore

In [ ]:
PATH = "encoding_chaining.csv"
df = pd.read_csv(PATH)
print(f"Raw shape: {df.shape}")
print(f"Columns  : {df.columns.tolist()}")
print(f"Subjects : {df['subject_id'].nunique()}")
print(f"Unique tasks   : {df['task'].unique()}")
print(f"Unique channels: {df['channel'].unique()}")
print(f"Unique subbands: {df['subband'].unique()}")
print(f"Unique features: {df['feature'].unique()}")
print(f"NaN count      : {df.isnull().sum().sum()}")
print(f"Zero chain_ratio: {(df['chain_ratio']==0).sum()}")


## 2. Pivot to Wide Format

In [ ]:
# Create a unique column key per (task, channel, subband, feature)
df['col_key'] = (df['task'] + '_' + df['channel'] + '_' +
                 df['subband'] + '_' + df['feature'])

pivot = df.pivot_table(
    index=['subject_id', 'scenario_id'],
    columns='col_key',
    values='chain_ratio',
    aggfunc='mean'       # mean in case of duplicates
).reset_index()

pivot.columns.name = None
print(f"Pivoted shape : {pivot.shape}")
print(f"Expected cols : 4 tasks × 3 channels × 4 subbands × 6 features = 288 feature cols")
print(f"Sample columns: {list(pivot.columns[:6])}")


In [ ]:
id_cols      = ['subject_id', 'scenario_id']
feature_cols = [c for c in pivot.columns if c not in id_cols]

# Zero values in chain_ratio are meaningful (no encoding activity) – keep them
# Only impute NaN
X_raw  = pivot[feature_cols].values.astype(float)
y      = pivot['scenario_id'].values - 1        # 0-indexed
groups = LabelEncoder().fit_transform(pivot['subject_id'])

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X_raw)

print(f"Feature matrix : {X.shape}")
print(f"Zeros preserved: {(X == 0).sum()}")
print(f"NaN remaining  : {np.isnan(X).sum()}")
print(f"Class balance  :\n{pd.Series(y).value_counts().sort_index().to_string()}")


## 3. Optional – PCA for Dimensionality Reduction
> Run this cell if `X.shape[1]` is very large (>200 cols) and training is slow.

In [ ]:
USE_PCA = False       # ← Set True to enable PCA reduction
PCA_COMPONENTS = 50   # ← Number of components to keep

if USE_PCA:
    from sklearn.preprocessing import StandardScaler
    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
    X = pca.fit_transform(X_scaled)
    print(f"PCA applied: {X.shape}  |  Variance explained: {pca.explained_variance_ratio_.sum():.3f}")
    feature_cols = [f"PC{i+1}" for i in range(PCA_COMPONENTS)]
else:
    print(f"PCA skipped – using all {X.shape[1]} features")


## 4. Train / Test Split (Group-based)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {X_train.shape} | subjects: {len(np.unique(groups[train_idx]))}")
print(f"Test : {X_test.shape}  | subjects: {len(np.unique(groups[test_idx]))}")
print("✅ Cross-subject split")


## 5. Define Model Pipelines

In [ ]:
models = {
    "Random Forest": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=15,
                                       random_state=42, n_jobs=-1))
    ]),
    "Logistic Regression": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs',
                                   random_state=42))
    ]),
    "KNN": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='mlogloss', random_state=42,
                              n_jobs=-1, tree_method='hist'))
    ]),
    "LightGBM": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                               random_state=42, n_jobs=-1, verbose=-1))
    ]),
    "AdaBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', AdaBoostClassifier(n_estimators=150, learning_rate=0.5,
                                   random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                           learning_rate=0.1, random_state=42))
    ]),
}

stk_estimators = [
    ('rf',   RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)),
    ('xgb',  XGBClassifier(n_estimators=150, tree_method='hist',
                            eval_metric='mlogloss', random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, random_state=42, verbose=-1)),
]
models["Stacking (RF+XGB+LGBM)"] = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', StackingClassifier(
        estimators=stk_estimators,
        final_estimator=LogisticRegression(max_iter=2000, random_state=42),
        cv=5, n_jobs=-1
    ))
])
print(f"Models: {list(models.keys())}")


## 6. Group K-Fold Cross-Validation

In [ ]:
N_SPLITS   = 10
gkf        = GroupKFold(n_splits=N_SPLITS)
cv_results = {}

print(f"{N_SPLITS}-Fold Group CV...\n")
for name, pipe in models.items():
    if "Stacking" in name:
        print(f"  {name:35s}: [skipped in CV]")
        continue
    res = cross_validate(pipe, X, y, cv=gkf, groups=groups,
                         scoring='accuracy', n_jobs=-1, return_train_score=True)
    cv_results[name] = res
    te = res['test_score']; tr = res['train_score']
    print(f"  {name:35s}: test={te.mean():.4f}±{te.std():.4f}  train={tr.mean():.4f}")


## 7. Train & Evaluate on Test Set

In [ ]:
test_results = {}
for name, pipe in models.items():
    print(f"  Fitting {name} ...", end='  ', flush=True)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    test_results[name] = {'accuracy': acc, 'f1_weighted': f1, 'y_pred': y_pred}
    print(f"Acc={acc:.4f}  F1={f1:.4f}")


## 8. Visualisations

In [ ]:
# FIG 1 – CV
cv_names = list(cv_results.keys())
cv_means = [cv_results[n]['test_score'].mean() for n in cv_names]
cv_stds  = [cv_results[n]['test_score'].std()  for n in cv_names]
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(cv_names, cv_means, xerr=cv_stds, height=0.5,
               color=plt.cm.Set2(np.linspace(0,1,len(cv_names))),
               alpha=0.85, error_kw=dict(ecolor='gray', capsize=4))
for bar, v in zip(bars, cv_means):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
ax.axvline(1/9, color='red', ls='--', alpha=0.5, label='Chance (11.1%)')
ax.set_xlabel('Accuracy'); ax.set_xlim(0, 1.1)
ax.set_title(f"{N_SPLITS}-Fold Group CV – Encoding Chaining (Cross-Subject)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_cv_group_kfold.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 2 – Test performance
nt = list(test_results.keys())
at = [test_results[n]['accuracy']    for n in nt]
ft = [test_results[n]['f1_weighted'] for n in nt]
x  = np.arange(len(nt)); w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x-w/2, at, w, label='Accuracy',    color='steelblue',  alpha=0.85)
b2 = ax.bar(x+w/2, ft, w, label='F1-Weighted', color='darkorange', alpha=0.85)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=8)
ax.axhline(1/9, color='red', ls='--', alpha=0.4, label='Chance')
ax.set_xticks(x); ax.set_xticklabels(nt, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.15)
ax.set_title("Test Performance – Encoding Chaining (Cross-Subject)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_test_performance.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 3 – Best CM
best_name = max(test_results, key=lambda n: test_results[n]['accuracy'])
print(f"Best: {best_name}  Acc={test_results[best_name]['accuracy']:.4f}")
cm = confusion_matrix(y_test, test_results[best_name]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix – {best_name}\n(Encoding, Cross-Subject)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_confusion_matrix_best.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 4 – Classification report
rpt = classification_report(y_test, test_results[best_name]['y_pred'], output_dict=True)
rdf = pd.DataFrame(rpt).T
print(classification_report(y_test, test_results[best_name]['y_pred'],
      target_names=[SCENARIO_LABELS[i] for i in range(9)]))


In [ ]:
# FIG 5 – RF Feature importance (top 30)
if not USE_PCA:
    rf_imp = models["Random Forest"].named_steps['clf'].feature_importances_
    fi = (pd.DataFrame({'feature': feature_cols, 'importance': rf_imp})
          .sort_values('importance', ascending=False).head(30))
    fig, ax = plt.subplots(figsize=(11, 9))
    ax.barh(fi['feature'], fi['importance'],
            color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 30))[::-1])
    ax.set_xlabel("Importance")
    ax.set_title("Top 30 Feature Importances – RF (Encoding Chaining)",
                 fontsize=12, fontweight='bold')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/05_rf_feature_importance.png", bbox_inches='tight')
    plt.show()
else:
    print("PCA was used – raw feature importance not available.")


In [ ]:
# FIG 6 – Stacking CM
stk_key = "Stacking (RF+XGB+LGBM)"
cm_s = confusion_matrix(y_test, test_results[stk_key]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix – Stacking (Encoding, Cross-Subject)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_stacking_confusion_matrix.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 7 – All CMs
nm = len(test_results); ncols = 4; nrows = (nm+ncols-1)//ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
fig.suptitle("All Confusion Matrices – Encoding Chaining", fontsize=13, fontweight='bold')
for idx, (name, res) in enumerate(test_results.items()):
    ax  = axes.flat[idx]
    cmi = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cmi, annot=True, fmt='d', cmap=CMAPS[idx%len(CMAPS)],
                ax=ax, xticklabels=LABELS_S, yticklabels=LABELS_S,
                linewidths=0.3, cbar=False)
    ax.set_title(f"{name}\nAcc={res['accuracy']:.3f}", fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)
for idx in range(nm, nrows*ncols): axes.flat[idx].axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_all_confusion_matrices.png", bbox_inches='tight')
plt.show()


## 9. Save Results

In [ ]:
summary = pd.DataFrame([{
    'Model':              n,
    'Test_Accuracy':      r['accuracy'],
    'F1_Weighted':        r['f1_weighted'],
    'GroupKFold_CV_Mean': cv_results[n]['test_score'].mean() if n in cv_results else None,
    'GroupKFold_CV_Std':  cv_results[n]['test_score'].std()  if n in cv_results else None,
} for n, r in test_results.items()]).sort_values('Test_Accuracy', ascending=False)

summary.to_csv(f"{OUTPUT_DIR}/model_results_summary.csv", index=False)
print(summary.to_string(index=False))
print(f"\n✅ Saved to {OUTPUT_DIR}/model_results_summary.csv")
